In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("data\\space_objects.csv")

In [4]:
def compute_semi_major_axis(mean_motion):
    mu = 398600
    n = mean_motion * 2 * np.pi / 86400
    return (mu / (n ** 2)) ** (1/3)

def generate_orbit(a, e, i, raan, argp, M0):
    t = np.linspace(0, 2*np.pi, 200)
    r = a * (1 - e**2) / (1 + e * np.cos(t))

    x = r * np.cos(t)
    y = r * np.sin(t)
    z = r * np.sin(np.radians(i))

    return np.vstack((x, y, z)).T

def compute_min_distance(p1, p2):
    return np.min(np.linalg.norm(p1 - p2, axis=1))

In [5]:
features = []
orbits = []

for _, row in df.iterrows():
    try:
        a = compute_semi_major_axis(row['MEAN_MOTION'])

        feat = [
            a,
            row['ECCENTRICITY'],
            row['INCLINATION'],
            row['RA_OF_ASC_NODE'],
            row['ARG_OF_PERICENTER'],
            row['MEAN_ANOMALY']
        ]

        orbit = generate_orbit(
            a,
            row['ECCENTRICITY'],
            row['INCLINATION'],
            row['RA_OF_ASC_NODE'],
            row['ARG_OF_PERICENTER'],
            row['MEAN_ANOMALY']
        )

        features.append(feat)
        orbits.append(orbit)

    except:
        continue

In [6]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

x = torch.tensor(features_scaled, dtype=torch.float)

In [7]:
k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(features_scaled)
distances, indices = nbrs.kneighbors(features_scaled)

edges = []
labels = []

for i in range(len(features_scaled)):
    for j_idx in range(1, k):

        j = indices[i][j_idx]

        # 🔥 REAL PHYSICS DISTANCE
        dist = compute_min_distance(orbits[i], orbits[j])

        edges.append([i, j])

        # REAL LABELS
        if dist < 25:
            labels.append(2)   # HIGH
        elif dist < 75:
            labels.append(1)   # MEDIUM
        else:
            labels.append(0)   # LOW

In [8]:
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
y = torch.tensor(labels, dtype=torch.long)

data = Data(x=x, edge_index=edge_index)

In [9]:
train_idx, test_idx = train_test_split(
    np.arange(len(y)),
    test_size=0.2,
    stratify=y.numpy(),
    random_state=42
)

train_idx = torch.tensor(train_idx)
test_idx = torch.tensor(test_idx)

In [13]:
class GATModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.gat1 = GATConv(6, 32)
        self.gat2 = GATConv(32, 16)
        self.fc = torch.nn.Linear(16, 3)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.gat1(x, edge_index)
        x = F.relu(x)

        x = self.gat2(x, edge_index)
        x = F.relu(x)

        return self.fc(x)

In [15]:
model = GATModel()

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

class_weights = torch.tensor([1.0, 3.0, 5.0])
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

for epoch in range(5000):

    model.train()
    optimizer.zero_grad()

    node_out = model(data)
    edge_out = node_out[edge_index[0]]

    loss = loss_fn(edge_out[train_idx], y[train_idx])

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.8172
Epoch 100, Loss: 0.5171
Epoch 200, Loss: 0.4932
Epoch 300, Loss: 0.4802
Epoch 400, Loss: 0.4715
Epoch 500, Loss: 0.4649
Epoch 600, Loss: 0.4601
Epoch 700, Loss: 0.4570
Epoch 800, Loss: 0.4547
Epoch 900, Loss: 0.4530
Epoch 1000, Loss: 0.4519
Epoch 1100, Loss: 0.4503
Epoch 1200, Loss: 0.4492
Epoch 1300, Loss: 0.4481
Epoch 1400, Loss: 0.4470
Epoch 1500, Loss: 0.4463
Epoch 1600, Loss: 0.4454
Epoch 1700, Loss: 0.4448
Epoch 1800, Loss: 0.4436
Epoch 1900, Loss: 0.4426
Epoch 2000, Loss: 0.4420
Epoch 2100, Loss: 0.4410
Epoch 2200, Loss: 0.4401
Epoch 2300, Loss: 0.4398
Epoch 2400, Loss: 0.4390
Epoch 2500, Loss: 0.4386
Epoch 2600, Loss: 0.4381
Epoch 2700, Loss: 0.4383
Epoch 2800, Loss: 0.4372
Epoch 2900, Loss: 0.4368
Epoch 3000, Loss: 0.4365
Epoch 3100, Loss: 0.4363
Epoch 3200, Loss: 0.4361
Epoch 3300, Loss: 0.4360
Epoch 3400, Loss: 0.4359
Epoch 3500, Loss: 0.4360
Epoch 3600, Loss: 0.4356
Epoch 3700, Loss: 0.4355
Epoch 3800, Loss: 0.4350
Epoch 3900, Loss: 0.4347
Epoch 4000, 

In [16]:
model.eval()

with torch.no_grad():
    node_out = model(data)
    edge_out = node_out[edge_index[0]]

    preds = torch.argmax(edge_out[test_idx], dim=1)

    correct = (preds == y[test_idx]).sum().item()
    acc = correct / len(test_idx)

print("Accuracy:", acc)

Accuracy: 0.6078026982899373


In [15]:
torch.save(model.state_dict(), "gat_modelll.pth")

In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [20]:
model.eval()

with torch.no_grad():
    node_out = model(data)              # node predictions
    edge_out = node_out[edge_index[0]]  # edge predictions

    probs = torch.softmax(edge_out, dim=1)
    preds = torch.argmax(probs, dim=1)

In [21]:
y_true = y.cpu().numpy()
y_pred = preds.cpu().numpy()
y_prob = probs.cpu().numpy()

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [23]:
acc = accuracy_score(y_true, y_pred)
print("Accuracy:", acc)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.6115942527155231
Precision: 0.6857029125611223
Recall: 0.6115942527155231
F1 Score: 0.49874066727260646


In [24]:
from collections import Counter
print(Counter(y.numpy()))

Counter({2: 33952, 0: 20490, 1: 3742})


In [25]:
class_weights = torch.tensor([1.0, 5.0, 10.0])
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [30]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y.numpy()),
    y=y.numpy()
)

class_weights = torch.tensor(weights, dtype=torch.float)
print("Class Weights:", class_weights)

Class Weights: tensor([0.9465, 5.1830, 0.5712])


In [31]:
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GATModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.gat1 = GATConv(6, 64, heads=4)
        self.gat2 = GATConv(64 * 4, 32)

        self.fc = torch.nn.Linear(32, 3)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.gat1(x, edge_index)
        x = F.elu(x)

        x = self.gat2(x, edge_index)
        x = F.elu(x)

        return self.fc(x)

In [32]:
model = GATModel()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [33]:
k = 10   # increase neighbors (previously 5)

In [34]:
for epoch in range(5000):

    model.train()
    optimizer.zero_grad()

    node_out = model(data)
    edge_out = node_out[edge_index[0]]

    loss = loss_fn(edge_out, y)

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.1068
Epoch 100, Loss: 1.0182
Epoch 200, Loss: 0.9726
Epoch 300, Loss: 0.9260
Epoch 400, Loss: 0.8980
Epoch 500, Loss: 0.8809
Epoch 600, Loss: 0.8693
Epoch 700, Loss: 0.8617
Epoch 800, Loss: 0.8557
Epoch 900, Loss: 0.8510
Epoch 1000, Loss: 0.8469
Epoch 1100, Loss: 0.8433
Epoch 1200, Loss: 0.8397
Epoch 1300, Loss: 0.8362
Epoch 1400, Loss: 0.8335
Epoch 1500, Loss: 0.8296
Epoch 1600, Loss: 0.8269
Epoch 1700, Loss: 0.8242
Epoch 1800, Loss: 0.8218
Epoch 1900, Loss: 0.8202
Epoch 2000, Loss: 0.8177
Epoch 2100, Loss: 0.8156
Epoch 2200, Loss: 0.8136
Epoch 2300, Loss: 0.8125
Epoch 2400, Loss: 0.8101
Epoch 2500, Loss: 0.8091
Epoch 2600, Loss: 0.8072
Epoch 2700, Loss: 0.8064
Epoch 2800, Loss: 0.8044
Epoch 2900, Loss: 0.8031
Epoch 3000, Loss: 0.8016
Epoch 3100, Loss: 0.8009
Epoch 3200, Loss: 0.7992
Epoch 3300, Loss: 0.7980
Epoch 3400, Loss: 0.7969
Epoch 3500, Loss: 0.7959
Epoch 3600, Loss: 0.7951
Epoch 3700, Loss: 0.7937
Epoch 3800, Loss: 0.7929
Epoch 3900, Loss: 0.7921
Epoch 4000, 

In [35]:
model.eval()

with torch.no_grad():
    node_out = model(data)
    edge_out = node_out[edge_index[0]]

    probs = torch.softmax(edge_out, dim=1)
    preds = torch.argmax(probs, dim=1)

y_true = y.cpu().numpy()
y_pred = preds.cpu().numpy()
y_prob = probs.cpu().numpy()

In [37]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='weighted'))
print("Recall:", recall_score(y_true, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_true, y_pred, average='weighted'))

# ROC AUC
roc = roc_auc_score(y_true, y_prob, multi_class='ovr')
print("ROC-AUC:", roc)

Accuracy: 0.5679568266190018
Precision: 0.6485520251249829
Recall: 0.5679568266190018
F1 Score: 0.5912040043705069
ROC-AUC: 0.7781882712201745
